# Cross-validation, part 3: all six models side by side

Reads the saved cross-validation predictions from `cv_trees.ipynb` and `cv_cnn.ipynb`. No training here.

1. One table: 6 models x (4-class, binary) x (accuracy, macro-F1, kappa), all 151 people
2. How much each score moves from fold to fold
3. Person by person: which model is better, with a 95% range
4. Per class, and for the 53+ age group
5. How the cross-validation compares with the single 30-person test of Stages 1 to 3

**Kernel:** `.venv-tf` in this folder.

## How to read this notebook (for beginners)

- **Pooled score:** all 151 people's windows put together, then scored once. The headline number.
- **Mean and spread over folds:** each of the 5 rounds is scored on its own ~30 people. A small spread means the score doesn't depend much on *which* people are tested.
- **Person by person:** each person gets their own kappa from each model. Then, for two models, we look at the **difference** person by person. If model A is better for most people, and the 95% range of the average difference stays above 0, A is genuinely better. If the range includes 0, the two are **statistically tied**: the gap could be luck.
- **95% range (bootstrap):** pick 151 people at random *with replacement* (some twice, some not at all), average their differences, and repeat 10,000 times. The middle 95% of those averages is the range.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from common import CLASSES, N_FOLDS, RESULTS_DIR, evaluate, paired_bootstrap, scores, to_binary

trees = np.load(RESULTS_DIR / 'cv_trees_predictions.npz')
cnn = np.load(RESULTS_DIR / 'cv_cnn_predictions.npz')
for key in ['pid', 'start', 'y', 'fold']:          # the same windows, in the same order, in the same folds
    assert (trees[key] == cnn[key]).all(), key
y, pid, fold = trees['y'], trees['pid'], trees['fold']
people = np.unique(pid)
assert len(people) == 151

MODELS = ['RF', 'RF+HMM', 'CNN', 'CNN+HMM', 'XGB', 'XGB+HMM']
pred = {n: trees[f'pred_{n}'] for n in ['RF', 'RF+HMM', 'XGB', 'XGB+HMM']}
pred.update({n: cnn[f'pred_{n}'] for n in ['CNN', 'CNN+HMM']})
pred = {n: pred[n] for n in MODELS}
print(f'{len(y):,} windows, {len(people)} people, {N_FOLDS} folds')

301,164 windows, 151 people, 5 folds


## 1. All 151 people pooled

In [2]:
pooled = pd.DataFrame([r for n in MODELS for r in evaluate(n, y, pred[n])])
pooled.to_csv(RESULTS_DIR / 'cv_metrics.csv', index=False)
print(pooled.pivot(index='model', columns='task', values=['accuracy', 'macro-F1', 'kappa']).loc[MODELS].round(3).to_string())

        accuracy        macro-F1          kappa       
task     4-class binary  4-class binary 4-class binary
model                                                 
RF         0.803  0.900    0.749  0.862   0.707  0.723
RF+HMM     0.883  0.929    0.822  0.899   0.825  0.799
CNN        0.788  0.912    0.718  0.876   0.683  0.751
CNN+HMM    0.859  0.930    0.781  0.901   0.788  0.802
XGB        0.802  0.901    0.744  0.865   0.707  0.730
XGB+HMM    0.881  0.932    0.821  0.906   0.822  0.812


## 2. Fold to fold

Mean and standard deviation of each score over the 5 folds.

In [3]:
per_fold = pd.DataFrame([{'fold': k, **r} for k in range(N_FOLDS) for n in MODELS
                         for r in evaluate(n, y[fold == k], pred[n][fold == k])])
per_fold.to_csv(RESULTS_DIR / 'cv_per_fold.csv', index=False)
spread = per_fold.groupby(['model', 'task'])[['accuracy', 'macro-F1', 'kappa']].agg(['mean', 'std'])
for task in ['4-class', 'binary']:
    t = spread.xs(task, level='task').loc[MODELS]
    print(f'===== {task} =====')
    print(pd.DataFrame({m: t[m]['mean'].map('{:.3f}'.format) + ' ± ' + t[m]['std'].map('{:.3f}'.format)
                        for m in ['accuracy', 'macro-F1', 'kappa']}).to_string(), '\n')

===== 4-class =====
              accuracy       macro-F1          kappa
model                                               
RF       0.803 ± 0.013  0.749 ± 0.010  0.707 ± 0.015
RF+HMM   0.883 ± 0.011  0.821 ± 0.013  0.825 ± 0.013
CNN      0.788 ± 0.023  0.717 ± 0.015  0.683 ± 0.028
CNN+HMM  0.859 ± 0.011  0.777 ± 0.013  0.787 ± 0.014
XGB      0.802 ± 0.014  0.743 ± 0.009  0.707 ± 0.016
XGB+HMM  0.881 ± 0.014  0.820 ± 0.008  0.822 ± 0.018 

===== binary =====
              accuracy       macro-F1          kappa
model                                               
RF       0.900 ± 0.002  0.861 ± 0.009  0.721 ± 0.019
RF+HMM   0.929 ± 0.003  0.899 ± 0.006  0.797 ± 0.012
CNN      0.912 ± 0.007  0.874 ± 0.016  0.747 ± 0.031
CNN+HMM  0.930 ± 0.006  0.899 ± 0.017  0.798 ± 0.034
XGB      0.901 ± 0.001  0.864 ± 0.010  0.728 ± 0.019
XGB+HMM  0.932 ± 0.004  0.905 ± 0.007  0.811 ± 0.014 



## 3. Person by person

Kappa for each person and model. A person whose day has only one class in it can't get a kappa (it is 0/0), so such people are left out of that comparison; the count is printed.

In [4]:
rows = []
for p in people:
    m = pid == p
    row = {'pid': p, 'fold': int(fold[m][0])}
    for n in MODELS:
        row[f'{n} 4-class'] = scores(y[m], pred[n][m], 4)['kappa']
        row[f'{n} binary'] = scores(to_binary(y[m]), to_binary(pred[n][m]), 2)['kappa']
    rows.append(row)
per_person = pd.DataFrame(rows)
ages = pd.read_csv('../../data/open datasets/capture24/capture24/metadata.csv').set_index('pid')['age']
per_person['age'] = ages[per_person.pid].values
per_person.to_csv(RESULTS_DIR / 'cv_per_person.csv', index=False)

print('median kappa per person:')
print(pd.DataFrame({t: {n: per_person[f'{n} {t}'].median() for n in MODELS} for t in ['4-class', 'binary']}).round(3).to_string())

pairs = [('XGB+HMM', 'RF+HMM'), ('RF+HMM', 'CNN+HMM'), ('XGB+HMM', 'CNN+HMM'),
         ('RF+HMM', 'RF'), ('CNN+HMM', 'CNN'), ('XGB+HMM', 'XGB')]
rows = []
for a, b in pairs:
    for t in ['4-class', 'binary']:
        both = per_person[[f'{a} {t}', f'{b} {t}']].dropna()
        mean, lo, hi = paired_bootstrap(both.iloc[:, 0], both.iloc[:, 1])
        d = both.iloc[:, 0] - both.iloc[:, 1]
        rows.append({'A': a, 'B': b, 'task': t, 'people': len(both), 'A better for': int((d > 0).sum()),
                     'mean kappa A - B': mean, '95% low': lo, '95% high': hi,
                     'verdict': 'A better' if lo > 0 else ('B better' if hi < 0 else 'tie')})
pairwise = pd.DataFrame(rows)
pairwise.to_csv(RESULTS_DIR / 'cv_pairwise.csv', index=False)
print()
print(pairwise.round(4).to_string(index=False))

median kappa per person:
         4-class  binary
RF         0.705   0.718
RF+HMM     0.835   0.807
CNN        0.683   0.748
CNN+HMM    0.797   0.799
XGB        0.703   0.726
XGB+HMM    0.834   0.834

      A       B    task  people  A better for  mean kappa A - B  95% low  95% high  verdict
XGB+HMM  RF+HMM 4-class     151            80           -0.0039  -0.0101    0.0021      tie
XGB+HMM  RF+HMM  binary     151            94            0.0190   0.0096    0.0297 A better
 RF+HMM CNN+HMM 4-class     151           105            0.0392   0.0276    0.0513 A better
 RF+HMM CNN+HMM  binary     151            79           -0.0014  -0.0147    0.0117      tie
XGB+HMM CNN+HMM 4-class     151           100            0.0353   0.0227    0.0483 A better
XGB+HMM CNN+HMM  binary     151            92            0.0176   0.0027    0.0330 A better
 RF+HMM      RF 4-class     151           151            0.1226   0.1152    0.1300 A better
 RF+HMM      RF  binary     151           138            0.0779

## 4. Per class, and the oldest group

F1 for each class (pooled), and the median 4-class kappa for people aged 53+, the group closest to SilverGuard's users.

In [5]:
per_class = pd.DataFrame({n: f1_score(y, pred[n], labels=range(4), average=None) for n in MODELS}, index=CLASSES).T
print('F1 per class, all 151 people:')
print(per_class.loc[MODELS].round(3).to_string())

print()
age_table = pd.DataFrame({label: {n: per_person.loc[mask, f'{n} 4-class'].median() for n in MODELS}
                          for label, mask in [(f'53+ (n={(per_person.age == "53+").sum()})', per_person.age == '53+'),
                                              (f'under 53 (n={(per_person.age != "53+").sum()})', per_person.age != '53+')]})
print('median 4-class kappa per person, by age:')
print(age_table.round(3).to_string())

F1 per class, all 151 people:
         sleep  sedentary  light  moderate-vigorous
RF       0.904      0.798  0.666              0.629
RF+HMM   0.971      0.886  0.743              0.688
CNN      0.884      0.777  0.666              0.544
CNN+HMM  0.947      0.863  0.727              0.588
XGB      0.904      0.796  0.671              0.603
XGB+HMM  0.964      0.883  0.761              0.675

median 4-class kappa per person, by age:
         53+ (n=34)  under 53 (n=117)
RF            0.700             0.710
RF+HMM        0.820             0.840
CNN           0.636             0.697
CNN+HMM       0.737             0.799
XGB           0.687             0.706
XGB+HMM       0.831             0.834


## 5. Cross-validation vs the single 30-person test

The Stage 1 to 3 numbers came from one split. If the two agree, the single split was not a lucky or unlucky draw.

In [6]:
single = pd.concat([pd.read_csv(RESULTS_DIR / f) for f in ['stage1_metrics.csv', 'stage2_metrics.csv', 'stage3_xgb_metrics.csv']])
single = single[single.model.isin(MODELS)].set_index(['model', 'task'])['kappa']
cv = pooled.set_index(['model', 'task'])['kappa']
both = pd.DataFrame({'single 30-person test': single, 'cross-validation, 151 people': cv})
both['difference'] = both.iloc[:, 1] - both.iloc[:, 0]
print('kappa:')
print(both.unstack('task').reindex(MODELS).round(3).to_string())

kappa:
        single 30-person test        cross-validation, 151 people        difference       
task                  4-class binary                      4-class binary    4-class binary
model                                                                                     
RF                      0.701  0.705                        0.707  0.723      0.006  0.018
RF+HMM                  0.814  0.778                        0.825  0.799      0.011  0.021
CNN                     0.659  0.705                        0.683  0.751      0.024  0.046
CNN+HMM                 0.768  0.758                        0.788  0.802      0.019  0.044
XGB                     0.696  0.697                        0.707  0.730      0.011  0.032
XGB+HMM                 0.818  0.790                        0.822  0.812      0.004  0.022
